# 1. Setup

In [2]:
import pandas as pd
import numpy as np 

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV,RandomizedSearchCV
from sklearn.base import clone
from scipy.stats import rankdata,spearmanr



In [4]:
RANDOM_STATE = 42
TARGET = "is_fraud"

In [5]:
def load_data():
    train = pd.read_csv("../data/raw/train.csv")
    test = pd.read_csv("../data/raw/test.csv")
    orig = pd.read_csv("../data/raw/credit_card_fraud_2026.csv")
    return train,test,orig

In [6]:
pd.set_option("display.max_columns",None)
train,test,orig = load_data()

## Column Descriptions

| Column | Type | Description |
|---|---|---|
| `id` | ID | Row identifier. Not a feature; drop before training. |
| `amount_usd` | Numeric | Transaction amount in US dollars. |
| `hours_since_last_txn` | Numeric | Hours since the card's previous transaction. Very small values suggest rapid repeated use. |
| `txn_count_last_24h` | Numeric | Number of transactions on the card in the last 24 hours. |
| `distance_from_home_km` | Numeric | Distance between the transaction location and the customer's home, in km. |
| `card_age_months` | Numeric | How long the card has existed, in months. |
| `customer_age` | Numeric | Age of the cardholder in years. |
| `account_balance_usd` | Numeric | Account balance in US dollars at the time of the transaction. |
| `cvv_retry_count` | Numeric | Number of wrong CVV (security code) entries before this transaction. |
| `velocity_score` | Numeric | Summary score of how fast and how much the card is being used. Higher means more unusual activity. |
| `time_of_day_hour` | Numeric | Hour of the transaction (0–23). EDA showed no meaningful link to fraud. |
| `day_of_week` | Categorical | Day of the week of the transaction. |
| `merchant_risk_score` | Numeric | Risk score assigned to the merchant. Higher means riskier. |
| `prior_disputes` | Numeric | Number of past transactions the customer disputed. |
| `merchant_category` | Categorical | Type of business (e.g. grocery, electronics, travel). |
| `card_type` | Categorical | Type of card (e.g. credit, debit, prepaid). |
| `auth_method` | Categorical | How the transaction was verified (e.g. PIN, OTP, 3D Secure). |
| `channel` | Categorical | Where the transaction happened (POS, Online, In-App). |
| `device_type` | Categorical | Device used (e.g. iPhone, Android Phone, Tablet, Windows PC). |
| `is_foreign_transaction` | Boolean | True if the transaction is in a different country from the card's home country. |
| `is_new_merchant` | Boolean | True if the customer has never used this merchant before. |
| `used_vpn` | Boolean | True if the connection went through a VPN, which can hide the real location. |
| `ip_country_mismatch` | Boolean | True if the IP address country differs from the card's or customer's country. |
| `billing_shipping_mismatch` | Boolean | True if the billing and shipping addresses differ. |
| `is_ai_generated_scam_attempt` | Boolean | True if the transaction was flagged as part of an AI-generated scam (e.g. deepfake call, phishing). Check how strongly it predicts fraud. |
| `is_fraud` | Target | 1 if the transaction is fraud, 0 if legitimate. This is what the model predicts. |

In [7]:
def add_features(df):
    df = df.copy()
    df["amount_to_balance"] = df["amount_usd"] / (df["account_balance_usd"] + 1)
    df["risk_exposure"] = df["merchant_risk_score"] * df["amount_usd"]
    df["risk_count"] = df[["is_foreign_transaction", "is_new_merchant", "used_vpn",
                           "ip_country_mismatch", "billing_shipping_mismatch",
                           "is_ai_generated_scam_attempt"]].sum(axis=1)
    df["cvv_scam"] = df["cvv_retry_count"] * df["is_ai_generated_scam_attempt"]
    return df
train = add_features(train)
test = add_features(test)

In [8]:
orig["is_fraud"] = orig["is_fraud"].astype(int)
print("only in train",set(train.columns)-set(orig.columns))
print("only in orig", set(orig.columns) - set(train.columns))

only in train {'risk_exposure', 'id', 'cvv_scam', 'amount_to_balance', 'risk_count'}
only in orig {'transaction_id'}


In [26]:
train["source"] = 0
orig["source"] = 1
test["source"] = 2

In [27]:
def x_y(train):
    x = train.drop(columns=["id","is_fraud"])
    y = train["is_fraud"]
    return x,y
x,y = x_y(train)

In [ ]:
def encoder(df):
    df = df.copy()
    bool_cols = df.select_dtypes(include = "bool").columns
    cat_cols = df.select_dtypes(include=["object", "str"]).columns
    df[bool_cols] = df[bool_cols].astype(int)
    
    encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    df[cat_cols] = encoder.fit_transform(df[cat_cols])
    
    return df
x = encoder(x)

In [29]:
both = pd.concat([train,orig,test],ignore_index=True)
both = add_features(both)
both = encoder(both)
drop_cols = ["id","is_fraud","source"]

train_part = both[both["source"]==0]
orig_part = both[both["source"]==1]
test_part  = both[both["source"] == 2]

x = train_part.drop(columns=drop_cols).reset_index(drop=True)
y = train_part["is_fraud"].astype(int).reset_index(drop=True)

x_orig = orig_part.drop(columns=drop_cols)
y_orig = orig_part["is_fraud"].astype(int)

x_test = test_part.drop(columns=drop_cols)
test_ids = test_part["id"].astype(int).values


In [ ]:
class kfold():
    def __init__(self,n_split=5,
                  shuffle=True,
                  random_state=RANDOM_STATE):
        self.skf = StratifiedKFold(n_splits=n_split,
                                   shuffle=shuffle,
                                   random_state=random_state)
        
    def split(self,x,y):
        for train_idx,val_idx in self.skf.split(x,y):
            x_train,x_val = x.iloc[train_idx],x.iloc[val_idx]
            y_train,y_val = y.iloc[train_idx],y.iloc[val_idx]
            yield x_train, x_val, y_train, y_val
kf5 = kfold()

In [14]:
models = {
    "LGBMClassifier":LGBMClassifier(n_estimators=3000,
                                    learning_rate=0.07,
                                    num_leaves=4,
                                    max_depth=4,
                                    verbose=-1,
                                    random_state=RANDOM_STATE)
}

In [16]:
def cv_engine(x, y, models, kf):
    results = []
    for name, model in models.items():
        fold_scores = []
        for x_tr, x_va, y_tr, y_va in kf.split(x, y):
            m = clone(model)                      # fresh, untrained copy for each fold
            m.fit(x_tr, y_tr)
            proba = m.predict_proba(x_va)[:, 1]
            fold_scores.append(roc_auc_score(y_va, proba))
        results.append({
            "model_name": name,
            "cv_mean": np.mean(fold_scores),
            "cv_std": np.std(fold_scores),
        })
        print(name, [round(s, 5) for s in fold_scores])
    return pd.DataFrame(results).sort_values("cv_mean", ascending=False)

cv_engine(x, y, models, kf5)

LGBMClassifier [0.94513, 0.9486, 0.95127, 0.94691, 0.94269]


,model_name,cv_mean,cv_std
0,LGBMClassifier,0.946921,0.002927


In [19]:
def tune(x,y,kf):
    param_distributions = {
        "learning_rate":[0.01,0.03,0.05,0.07],
        "num_leaves":[2,4,7,9],
        "max_depth":[2,4,6,7],
        "min_child_samples":[20,50,70,100]}    

    search = RandomizedSearchCV(
        estimator = LGBMClassifier(n_estimators=3000,
                                   n_jobs=1,
                                  verbose = -1,
                                  random_state = RANDOM_STATE),
        param_distributions = param_distributions,
        n_iter = 50,
        scoring = "roc_auc",
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        random_state= RANDOM_STATE,
        n_jobs=-1)

    search.fit(x,y)        
    print("Best params:", search.best_params_)
    print("Best CV score:", search.best_score_)
    return search.best_estimator_

In [22]:
best_model = tune(x, y, kf5)
best_model

Best params: {'num_leaves': 4, 'min_child_samples': 70, 'max_depth': 2, 'learning_rate': 0.05}
Best CV score: 0.9477129349344079


,num_leaves,4
,max_depth,2
,learning_rate,0.05
,n_estimators,3000
,min_child_samples,70
,random_state,42
,n_jobs,1
,verbose,-1
,boosting_type,'gbdt'
,subsample_for_bin,200000
,objective,None


In [ ]:
model_tuned = {
    "LGBMClassifier":LGBMClassifier(n_estimators=3000,
                                    learning_rate=0.05,
                                    num_leaves=4,
                                    max_depth=2,
                                    min_child_samples=70,
                                    verbose=-1,
                                    random_state=RANDOM_STATE)
}

In [23]:
def tune2(x,y,kf):
    param_distributions = {
        "max_depth": [2, 3, 4],
        "learning_rate": [0.03, 0.05, 0.07],
        "min_child_weight": [1, 5, 10],
        "subsample": [0.7, 0.85, 1.0],
        "colsample_bytree": [0.7, 0.85, 1.0],}  

    search = RandomizedSearchCV(
        estimator=XGBClassifier(n_estimators=3000,
                                n_jobs=1,
                                verbosity=0,
                                random_state=RANDOM_STATE),
        param_distributions = param_distributions,
        n_iter = 50,
        scoring = "roc_auc",
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        random_state= RANDOM_STATE,
        n_jobs=-1)

    search.fit(x,y)        
    print("Best params:", search.best_params_)
    print("Best CV score:", search.best_score_)
    return search.best_estimator_

In [24]:
best_model_2 = tune2(x,y,kf5)
best_model_2

Best params: {'subsample': 1.0, 'min_child_weight': 5, 'max_depth': 2, 'learning_rate': 0.05, 'colsample_bytree': 0.85}
Best CV score: 0.9475505853909377


,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.85
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [15]:
models = {
    "lgbm": LGBMClassifier(n_estimators=3000, learning_rate=0.05, num_leaves=4, max_depth=2,
                           min_child_samples=70, n_jobs=-1, verbose=-1,
                           random_state=RANDOM_STATE),
    "xgb": XGBClassifier(n_estimators=3000, learning_rate=0.05, max_depth=2,
                         min_child_weight=5, subsample=1.0, colsample_bytree=0.85,
                         n_jobs=-1, verbosity=0, random_state=RANDOM_STATE),
}

In [16]:
model2_tuned = {
    "XGBClassifier":XGBClassifier(n_estimators=3000,
                                    learning_rate=0.05,
                                    subsample=1.0,
                                    max_depth=2,
                                    min_child_weight=5,
                                    colsample_bytree=0.85,
                                    verbose=-1,
                                    random_state=RANDOM_STATE)
}

In [17]:
models = {
    "lgbm": LGBMClassifier(n_estimators=3000, learning_rate=0.05, num_leaves=4, max_depth=2,
                           min_child_samples=70, n_jobs=-1, verbose=-1,
                           random_state=RANDOM_STATE),
    "xgb": XGBClassifier(n_estimators=3000, learning_rate=0.05, max_depth=2,
                         min_child_weight=5, subsample=1.0, colsample_bytree=0.85,
                         n_jobs=-1, verbosity=0, random_state=RANDOM_STATE)}

In [22]:
def get_oof(model,x,y,kf,x_extra=None,y_extra=None):
    oof = np.zeros(len(y))
    for train_idx,val_idx in kf.skf.split(x,y):
        x_tr,y_tr = x.iloc[train_idx],y.iloc[train_idx]
        if x_extra is not None:
            x_tr = pd.concat([x_tr,x_extra])
            y_tr = pd.concat([y_tr,y_extra])
        m = clone(model)
        m.fit(x_tr,y_tr)
        oof[val_idx] = m.predict_proba(x.iloc[val_idx])[:,1]
    print(f"OOF AUC: {roc_auc_score(y, oof):.5f}")
    return oof

In [30]:
oofs = {}
for name,model in models.items():
    print(f"--- {name} without orig ---")
    oofs[f"{name}_base"] = get_oof(model, x, y, kf5)
    print(f"--- {name} with orig ---")
    oofs[f"{name}_orig"] = get_oof(model, x, y, kf5, x_orig, y_orig)

--- lgbm without orig ---
OOF AUC: 0.94768
--- lgbm with orig ---
OOF AUC: 0.94755
--- xgb without orig ---
OOF AUC: 0.94751
--- xgb with orig ---
OOF AUC: 0.94756


In [31]:
print(x_orig.shape, y_orig.shape)
print(both["source"].value_counts())

(20000, 30) (20000,)
source
0    699960
2    299984
1     20000
Name: count, dtype: int64


In [36]:
def blend_score(names):
    blend = np.mean([rankdata(oofs[n]) for n in names],axis=0)
    return roc_auc_score(y, blend)

print("Correlation:",spearmanr(oofs["lgbm_base"],oofs["xgb_base"]).correlation)
print(f"Blend base: {blend_score(['lgbm_base', 'xgb_base']):.5f}")
print(f"Blend orig: {blend_score(['lgbm_orig', 'xgb_orig']):.5f}")


Correlation: 0.9950773282750089
Blend base: 0.94773
Blend orig: 0.94766


In [37]:
from scipy.stats import rankdata

USE_ORIG = False   # the blend without the original data scored higher

x_full = pd.concat([x, x_orig]) if USE_ORIG else x
y_full = pd.concat([y, y_orig]) if USE_ORIG else y

# 1. Train each model on ALL training data and predict the test set
test_preds = []
for name, model in models.items():
    print(f"Training {name} on full data...")
    m = clone(model)
    m.fit(x_full, y_full)
    p = m.predict_proba(x_test)[:, 1]
    test_preds.append(rankdata(p) / len(p))   # ranks scaled to 0–1

# 2. Blend: average the two models per transaction
blend = np.mean(test_preds, axis=0)

# 3. Build the submission file
submission = pd.DataFrame({"id": test_ids, "is_fraud": blend})
submission.to_csv("submission.csv", index=False)

# 4. Check it matches Kaggle's expected format
sample = pd.read_csv("../data/raw/sample_submission.csv")
print("Shapes:", sample.shape, submission.shape)
print("Columns:", sample.columns.tolist(), submission.columns.tolist())
print("Same ids:", (sample["id"].values == submission["id"].values).all())
submission.head()

Training lgbm on full data...
Training xgb on full data...
Shapes: (299984, 2) (299984, 2)
Columns: ['id', 'is_fraud'] ['id', 'is_fraud']
Same ids: True


,id,is_fraud
0,699960,0.148616
1,699961,0.055415
2,699962,0.950056
3,699963,0.411559
4,699964,0.391533
